# 01. Prepare the data

This notebook turns the two raw files into one clean table that every later notebook reads.

| | File | Content |
|---|---|---|
| Input | `ods032.parquet` | Elia solar output, forecasts and capacity, every 15 minutes |
| Input | `weather_belgium.parquet` | Open-Meteo weather, hourly, 11 areas |
| Output | `model_table.parquet` | One row per area per hour, from January 2023: target, raw features, benchmark |

**Rules followed here**
- Nothing is learned from the data in this notebook (no averages, no scaling), so nothing can leak from the test period. Missing values are left missing, and are filled later using training data only.
- Every step that removes rows is logged, and the log is shown at the end.

**Steps**
1. Elia: select, convert, aggregate, build the target
2. Weather: blank out bad values
3. Join
4. Flags: daylight and train/test split
5. Final checks and save

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ELIA_FILE = Path("ods032.parquet")
WEATHER_FILE = Path("weather_belgium.parquet")
OUT_FILE = Path("model_table.parquet")

SAMPLE_START = "2023-01-01"     # first day of the modelling sample (UTC)
TEST_START = "2025-01-01"       # rows from this day on are the test set; earlier rows are for training and tuning

pd.set_option("display.float_format", "{:,.2f}".format)

log = []                        # one entry per step: what was done and how many rows remain
def record(step, rows, note=""):
    removed = (log[-1]["rows"] - rows) if log and log[-1]["table"] == step.split(":")[0] else 0
    log.append({"table": step.split(":")[0], "step": step.split(":", 1)[1].strip(), "rows": rows,
                "removed": removed, "note": note})
    print(f"{step:<55s} {rows:>10,} rows" + (f"   (removed {removed:,})" if removed else ""))

## 1. Elia

### 1.1 Load and select columns

Only the columns that are safe to use are read. Three groups are left out:

| Left out | Why |
|---|---|
| `loadfactor` | It is the target divided by capacity, so it would give the answer away |
| Most recent forecast (3 columns) | Issued just before delivery, long after the trading decision |
| Latest day-ahead forecast (3 columns) | Updated after the 11:00 decision time |
| Week-ahead forecast (3 columns), `resolutioncode` | Not needed |

Elia's 11:00 day-ahead forecast is kept as the **benchmark**, not as a feature.

In [2]:
keep = ["datetime", "region", "measured", "monitoredcapacity",
        "dayahead11hforecast", "dayahead11hconfidence10", "dayahead11hconfidence90"]
elia = pd.read_parquet(ELIA_FILE, columns=keep)
record("Elia: raw file", len(elia))

Elia: raw file                                           2,978,360 rows


### 1.2 Provinces only, UTC, from January 2023

- **Provinces only:** Belgium, Flanders and Wallonia are exact sums of the provinces. Keeping them would count the same output twice.
- **UTC:** the weather file is in UTC, so both tables must share one clock.
- **January 2023:** before spring 2022 Elia updated capacity late, which makes early load factors unreliable.

In [3]:
AGGREGATES = ["Belgium", "Flanders", "Wallonia"]
elia = elia[~elia["region"].isin(AGGREGATES)]
record("Elia: drop Belgium, Flanders, Wallonia", len(elia))

elia["datetime_utc"] = pd.to_datetime(elia["datetime"], utc=True)
elia = elia.drop(columns="datetime")
elia = elia[elia["datetime_utc"] >= pd.Timestamp(SAMPLE_START, tz="UTC")]
record(f"Elia: keep from {SAMPLE_START}", len(elia))

dups = elia.duplicated(["region", "datetime_utc"]).sum()
elia = elia.drop_duplicates(["region", "datetime_utc"])
record("Elia: drop duplicate (area, time) rows", len(elia), f"{dups} duplicates")
print(f"Areas: {sorted(elia['region'].unique())}")

Elia: drop Belgium, Flanders, Wallonia                   2,340,140 rows   (removed 638,220)
Elia: keep from 2023-01-01                               1,472,020 rows   (removed 868,120)


Elia: drop duplicate (area, time) rows                   1,472,020 rows
Areas: ['Antwerp', 'Brussels', 'East-Flanders', 'Flemish-Brabant', 'Hainaut', 'Limburg', 'Liège', 'Luxembourg', 'Namur', 'Walloon-Brabant', 'West-Flanders']


### 1.3 From 15 minutes to hourly

Weather is hourly, so the four quarter-hour values of each hour are averaged.

Two details matter:
- **Labelling.** Open-Meteo's radiation at hour *T* is the average over the hour ending at *T*. Each Elia hour is therefore labelled with its **end** time, so both tables describe the same 60 minutes.
- **Completeness.** An hour is kept if all four quarter-hour values of measured output are present. Averaging three out of four would bias a daytime hour. One exception: if a value is missing but every other value in that hour is zero, the hour is night and is kept as zero. In this file the missing values are all of that kind, the 23:45 (local time) reading on days up to October 2024.

In [4]:
missing_15 = elia["measured"].isna()
print(f"Quarter-hour rows with missing measured output: {missing_15.sum():,} ({missing_15.mean():.2%})")
local = elia.loc[missing_15, "datetime_utc"].dt.tz_convert("Europe/Brussels")
print(f"  local times of the missing values: {local.dt.strftime('%H:%M').value_counts().to_dict()}")
print(f"  from {local.min():%d %b %Y} to {local.max():%d %b %Y}")

value_cols = ["measured", "monitoredcapacity", "dayahead11hforecast",
              "dayahead11hconfidence10", "dayahead11hconfidence90"]
grouped = (elia.set_index("datetime_utc").groupby("region")[value_cols]
           .resample("1h", label="right", closed="left"))
hourly = grouped.mean()
n_valid = grouped["measured"].count()["measured"]      # how many of the four values exist
max_present = grouped["measured"].max()["measured"]    # largest of the values that exist
record("Elia: aggregate to hourly", len(hourly))

complete = n_valid == 4
night_gap = (n_valid < 4) & (n_valid > 0) & (max_present == 0)
print(f"Hours with a missing value: {(~complete).sum():,}, of which night hours kept as zero: {night_gap.sum():,}")
hourly = hourly[complete | night_gap].reset_index()
record("Elia: drop incomplete daytime hours", len(hourly), f"{night_gap.sum()} night hours with one missing value kept as zero")

Quarter-hour rows with missing measured output: 6,512 (0.44%)
  local times of the missing values: {'23:45': 6512}
  from 01 Jan 2023 to 15 Oct 2024


Elia: aggregate to hourly                                  368,005 rows   (removed 1,104,015)
Hours with a missing value: 6,512, of which night hours kept as zero: 6,512
Elia: drop incomplete daytime hours                        368,005 rows


### 1.4 Build the target

The target is the **load factor**: measured output divided by monitored capacity, a number between 0 and 1.

Why not predict MW directly? Installed capacity nearly doubles over the sample. In MW, the same sunny afternoon produces far more in 2026 than in 2023, for a reason unrelated to weather. Dividing by capacity removes that trend, so all years are comparable. A forecast in MW is recovered by multiplying back by capacity.

The benchmark columns are converted the same way.

In [5]:
hourly["lf"] = hourly["measured"] / hourly["monitoredcapacity"]
hourly["bench_elia_lf"] = hourly["dayahead11hforecast"] / hourly["monitoredcapacity"]
hourly["bench_elia_p10_lf"] = hourly["dayahead11hconfidence10"] / hourly["monitoredcapacity"]
hourly["bench_elia_p90_lf"] = hourly["dayahead11hconfidence90"] / hourly["monitoredcapacity"]

impossible = (hourly["lf"] < 0) | (hourly["lf"] > 1) | hourly["monitoredcapacity"].le(0)
print(f"Hours with a load factor below 0 or above 1: {impossible.sum()}")
hourly = hourly[~impossible]
record("Elia: drop impossible load factors", len(hourly))

hourly = hourly.rename(columns={"measured": "measured_mw", "monitoredcapacity": "capacity_mw"})
hourly = hourly.drop(columns=["dayahead11hforecast", "dayahead11hconfidence10", "dayahead11hconfidence90"])
print(f"Highest load factor: {hourly['lf'].max():.1%}   Benchmark missing in {hourly['bench_elia_lf'].isna().mean():.2%} of hours")

Hours with a load factor below 0 or above 1: 0
Elia: drop impossible load factors                         368,005 rows
Highest load factor: 78.9%   Benchmark missing in 0.07% of hours


## 2. Weather

### 2.1 Load and restrict to the sample period

In [6]:
wx = pd.read_parquet(WEATHER_FILE).drop(columns=["latitude", "longitude"])
record("Weather: raw file", len(wx))
wx = wx[wx["datetime_utc"] >= pd.Timestamp(SAMPLE_START, tz="UTC")]
record(f"Weather: keep from {SAMPLE_START}", len(wx))

WEATHER_FEATURES = [c for c in wx.columns if c not in ("datetime_utc", "region")]
print(f"{len(WEATHER_FEATURES)} weather features: {WEATHER_FEATURES}")

Weather: raw file                                          560,736 rows
Weather: keep from 2023-01-01                              368,016 rows   (removed 192,720)
16 weather features: ['ghi_wm2', 'direct_wm2', 'diffuse_wm2', 'dni_wm2', 'cloud_total_pct', 'cloud_low_pct', 'cloud_mid_pct', 'cloud_high_pct', 'temp_c', 'dew_point_c', 'humidity_pct', 'pressure_hpa', 'precip_mm', 'wind_kmh', 'wind_gust_kmh', 'wind_dir_deg']


### 2.2 Blank out implausible values

Each variable has a range that is physically plausible for Belgium. A value outside it is set to missing. The row is kept, since its other variables are still good.

The known case is 26 to 28 June 2024, when relative humidity and dew point are corrupted in every area.

In [7]:
LIMITS = {   # column: (lowest plausible, highest plausible)
    "ghi_wm2": (0, 1100), "direct_wm2": (0, 1000), "diffuse_wm2": (0, 700), "dni_wm2": (0, 1100),
    "cloud_total_pct": (0, 100), "cloud_low_pct": (0, 100), "cloud_mid_pct": (0, 100), "cloud_high_pct": (0, 100),
    "temp_c": (-25, 45), "dew_point_c": (-25, 30), "humidity_pct": (10, 100), "pressure_hpa": (900, 1060),
    "precip_mm": (0, 60), "wind_kmh": (0, 130), "wind_gust_kmh": (0, 200), "wind_dir_deg": (0, 360),
}
blanked = {}
for c in WEATHER_FEATURES:
    lo, hi = LIMITS[c]
    bad = (wx[c] < lo) | (wx[c] > hi)
    if bad.any():
        blanked[c] = {"values blanked": int(bad.sum()),
                      "first": wx.loc[bad, "datetime_utc"].min().strftime("%Y-%m-%d"),
                      "last": wx.loc[bad, "datetime_utc"].max().strftime("%Y-%m-%d")}
        wx.loc[bad, c] = np.nan
record("Weather: blank implausible values (rows kept)", len(wx), f"{sum(v['values blanked'] for v in blanked.values())} values")
pd.DataFrame(blanked).T if blanked else "No implausible values"

Weather: blank implausible values (rows kept)              368,016 rows


,values blanked,first,last
dew_point_c,660,2024-06-26,2024-06-28
humidity_pct,660,2024-06-26,2024-06-28


### 2.3 A known weakness that is kept, not fixed

From mid 2024, wind speed, wind gusts and mid-level cloud cover sit at a lower level, which points to a change in the weather model behind the archive (see `explore_weather.ipynb`, section 4.4). The training years and the test years therefore do not measure these three variables in quite the same way.

They are kept, because they are raw features of the registered dataset. They are listed here so that the modelling notebook can test whether results depend on them.

In [8]:
SHIFTED_FEATURES = ["wind_kmh", "wind_gust_kmh", "cloud_mid_pct"]
before = wx[wx["datetime_utc"] < "2024-07-01"][SHIFTED_FEATURES].mean()
after = wx[wx["datetime_utc"] >= "2024-07-01"][SHIFTED_FEATURES].mean()
pd.DataFrame({"mean before July 2024": before, "mean from July 2024": after, "change %": (after / before - 1) * 100})

,mean before July 2024,mean from July 2024,change %
wind_kmh,15.57,10.61,-31.85
wind_gust_kmh,29.52,20.77,-29.65
cloud_mid_pct,35.33,25.11,-28.94


## 3. Join

An inner join on area and hour: a row survives only if both the target and the weather exist for it.

In [9]:
table = hourly.merge(wx, on=["region", "datetime_utc"], how="inner", validate="one_to_one")
record("Joined: Elia hours matched with weather", len(table))
print(f"Elia hours without weather: {len(hourly) - len(table):,}")

Joined: Elia hours matched with weather                    368,005 rows
Elia hours without weather: 0


## 4. Flags

### 4.1 Daylight

At night output is zero by definition, so there is nothing to forecast. Night rows are **kept** in the table, since later notebooks need them to build lagged features, but they are flagged so they can be left out of training and scoring.

The flag uses the weather forecast (irradiance above zero), which is known in advance. It must not use measured output, which is not.

In [10]:
table["is_daylight"] = table["ghi_wm2"] > 0

night = table[~table["is_daylight"]]
print(f"Daylight hours: {table['is_daylight'].sum():,} ({table['is_daylight'].mean():.1%} of rows)")
print(f"Share of all measured energy that falls in hours flagged as night: "
      f"{night['measured_mw'].sum() / table['measured_mw'].sum():.4%}")
print(f"Highest load factor in an hour flagged as night: {night['lf'].max():.2%}")

Daylight hours: 197,610 (53.7% of rows)
Share of all measured energy that falls in hours flagged as night: 0.0018%
Highest load factor in an hour flagged as night: 1.53%


### 4.2 Train and test

The split is by time, never random. A random split would let the model see days on both sides of each test day, which is information from the future.

- **train:** before 1 January 2025. All model fitting and tuning happens here (with time-ordered cross-validation inside it).
- **test:** from 1 January 2025. Looked at once, at the end.

In [11]:
table["split"] = np.where(table["datetime_utc"] < pd.Timestamp(TEST_START, tz="UTC"), "train", "test")
table.groupby("split").agg(rows=("lf", "size"), daylight_rows=("is_daylight", "sum"),
                           first=("datetime_utc", "min"), last=("datetime_utc", "max")).loc[["train", "test"]]

,rows,daylight_rows,first,last
split,,,,
train,192973,102323,2023-01-01 01:00:00+00:00,2024-12-31 23:00:00+00:00
test,175032,95287,2025-01-01 00:00:00+00:00,2026-10-25 23:00:00+00:00


## 5. Final checks and save

### 5.1 Column roles

In [12]:
RAW_FEATURES = WEATHER_FEATURES + ["region", "capacity_mw"]
roles = {"datetime_utc": "key", "region": "key and raw feature", "lf": "TARGET (load factor, 0 to 1)",
         "measured_mw": "target in MW, for reporting", "capacity_mw": "raw feature, and converts load factor to MW",
         "bench_elia_lf": "benchmark", "bench_elia_p10_lf": "benchmark", "bench_elia_p90_lf": "benchmark",
         "is_daylight": "flag", "split": "flag"}
roles.update({c: "raw feature (weather)" for c in WEATHER_FEATURES})

first = ["datetime_utc", "region", "split", "is_daylight", "lf", "measured_mw", "capacity_mw",
         "bench_elia_lf", "bench_elia_p10_lf", "bench_elia_p90_lf"]
table = table[first + WEATHER_FEATURES].sort_values(["region", "datetime_utc"]).reset_index(drop=True)

print(f"Raw features before any feature engineering: {len(RAW_FEATURES)} (the project requires at least 15)")
overview = pd.DataFrame({"role": [roles[c] for c in table.columns],
                         "missing %": [table[c].isna().mean() * 100 for c in table.columns]}, index=table.columns)
overview

Raw features before any feature engineering: 18 (the project requires at least 15)


,role,missing %
datetime_utc,key,0.00
region,key and raw feature,0.00
split,flag,0.00
is_daylight,flag,0.00
lf,"TARGET (load factor, 0 to 1)",0.00
measured_mw,"target in MW, for reporting",0.00
capacity_mw,"raw feature, and converts load factor to MW",0.00
bench_elia_lf,benchmark,0.07
bench_elia_p10_lf,benchmark,0.07
bench_elia_p90_lf,benchmark,0.07


### 5.2 Rows per area and coverage

In [13]:
expected = int((table["datetime_utc"].max() - table["datetime_utc"].min()) / pd.Timedelta("1h")) + 1
cov = table.groupby("region").agg(rows=("lf", "size"), first=("datetime_utc", "min"), last=("datetime_utc", "max"),
                                  mean_load_factor_pct=("lf", lambda s: s.mean() * 100))
cov["coverage %"] = cov["rows"] / expected * 100
cov

,rows,first,last,mean_load_factor_pct,coverage %
region,,,,,
Antwerp,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.68,100.00
Brussels,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.70,100.00
East-Flanders,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.92,100.00
Flemish-Brabant,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.71,100.00
Hainaut,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.65,100.00
Limburg,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.65,100.00
Liège,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.35,100.00
Luxembourg,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,9.92,100.00
Namur,33455,2023-01-01 01:00:00+00:00,2026-10-25 23:00:00+00:00,10.42,100.00


### 5.3 What each step removed

This table goes into the data section of the report.

In [14]:
steps = pd.DataFrame(log)[["table", "step", "rows", "removed", "note"]]
steps

,table,step,rows,removed,note
0,Elia,raw file,2978360,0,
1,Elia,"drop Belgium, Flanders, Wallonia",2340140,638220,
2,Elia,keep from 2023-01-01,1472020,868120,
3,Elia,"drop duplicate (area, time) rows",1472020,0,0 duplicates
4,Elia,aggregate to hourly,368005,1104015,
5,Elia,drop incomplete daytime hours,368005,0,6512 night hours with one missing value kept a...
6,Elia,drop impossible load factors,368005,0,
7,Weather,raw file,560736,0,
8,Weather,keep from 2023-01-01,368016,192720,
9,Weather,blank implausible values (rows kept),368016,0,1320 values


In [15]:
assert not table.duplicated(["region", "datetime_utc"]).any(), "duplicate rows"
assert table["lf"].between(0, 1).all(), "target outside 0 to 1"
assert table["lf"].notna().all(), "missing target"
assert (table["datetime_utc"] >= pd.Timestamp(SAMPLE_START, tz="UTC")).all()

table.to_parquet(OUT_FILE, index=False)
print(f"Saved {OUT_FILE}: {len(table):,} rows, {table.shape[1]} columns, {OUT_FILE.stat().st_size / 1e6:.1f} MB")
print(f"Period: {table['datetime_utc'].min()}  to  {table['datetime_utc'].max()}")

Saved model_table.parquet: 368,005 rows, 26 columns, 15.6 MB
Period: 2023-01-01 01:00:00+00:00  to  2026-10-25 23:00:00+00:00


## What the next notebook receives

`model_table.parquet`, one row per area per hour:

- **`lf`** is the target.
- **`bench_elia_*`** are the benchmark. They must never be used as model inputs.
- **`split`** and **`is_daylight`** say which rows to train on and which to score.
- **Weather columns, `region` and `capacity_mw`** are the raw features.
- **Missing values** in weather columns are still missing. Fill them using statistics computed on `split == "train"` only, or use a model that handles missing values natively.